In [ ]:
import stim
import pymatching
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
def count_logical_errors(circuit: stim.Circuit, num_shots: int) -> int:
    sampler = circuit.compile_detector_sampler()
    dets, obs = sampler.sample(num_shots, separate_observables=True)

    dem = circuit.detector_error_model(decompose_errors=True)
    matcher = pymatching.Matching.from_detector_error_model(dem)

    predictions = matcher.decode_batch(dets)

    # Count any logical mismatch
    return np.count_nonzero(np.any(predictions ^ obs, axis=1))


In [ ]:
def hamming_7_4_x_memory(p):
    c = stim.Circuit()

    # 7 data qubits: 0–6
    # 3 ancillas: 7–9
    data = list(range(7))
    anc = [7, 8, 9]

    # Initialize
    c.append("R", data + anc)

    # X noise on data
    # for q in data:
    #     c.append("X_ERROR", q, p)
    c.append("X_ERROR", data, p)

    #  Stabilizer 1: Z0 Z2 Z4 Z6 
    c.append("H", anc[0])
    for q in [0, 2, 4, 6]:
        c.append("CZ", [anc[0], q])
    c.append("H", anc[0])
    c.append("M", anc[0])
    c.append("DETECTOR", [stim.target_rec(-1)])

    #  Stabilizer 2: Z1 Z2 Z5 Z6 
    c.append("H", anc[1])
    for q in [1, 2, 5, 6]:
        c.append("CZ", [anc[1], q])
    c.append("H", anc[1])
    c.append("M", anc[1])
    c.append("DETECTOR", [stim.target_rec(-1)])

    #  Stabilizer 3: Z3 Z4 Z5 Z6 
    c.append("H", anc[2])
    for q in [3, 4, 5, 6]:
        c.append("CZ", [anc[2], q])
    c.append("H", anc[2])
    c.append("M", anc[2])
    c.append("DETECTOR", [stim.target_rec(-1)])

    # Measure data
    c.append("M", data)

    # Define a logical observable:
    # Logical X = parity of qubits [0,1,2]
    c.append(
        "OBSERVABLE_INCLUDE",
        [stim.target_rec(-7), stim.target_rec(-6), stim.target_rec(-5)],
        0
    )

    return c
c = hamming_7_4_x_memory(p = 0.01)
print(repr(c))
c.diagram('timeline-svg')

In [ ]:
num_shots = 100_000
num_logical_errors = count_logical_errors(c, num_shots)
print("there were", num_logical_errors, "wrong predictions (logical errors) out of", num_shots, "shots")

In [ ]:
import numpy as np
import stim
from IPython.display import display

# ----------------------------
# Hamming(7,4,3) Z-parity checks (detect X errors)
# ----------------------------
CHECKS = [
    [0, 2, 4, 6],
    [1, 2, 5, 6],
    [3, 4, 5, 6],
]

H = np.array([
    [1,0,1,0,1,0,1],
    [0,1,1,0,0,1,1],
    [0,0,0,1,1,1,1],
], dtype=np.uint8)

def make_syndrome_to_bitpos_map(H: np.ndarray):
    """Map a 3-bit syndrome to a single-bit error position (0..6)."""
    m, n = H.shape
    assert (m, n) == (3, 7)
    s2i = {tuple([0]*m): None}
    for j in range(n):
        s2i[tuple(H[:, j].tolist())] = j
    return s2i

S2I = make_syndrome_to_bitpos_map(H)

def append_x_only_syndrome_extraction(c: stim.Circuit, data: list[int], anc: list[int]):
    """
    Measure 3 Z-parity checks to detect X errors.
    Ancillas are reset, entangled via CX(data->anc), then measured in Z.
    """
    assert len(data) == 7 and len(anc) == 3
    c.append("R", anc)
    for r, qs in enumerate(CHECKS):
        a = anc[r]
        ops = []
        for q in qs:
            ops += [data[q], a]
        c.append("CX", ops)
    c.append("M", anc)

def build_two_block_demo_circuit(p_noise: float = 0.0) -> stim.Circuit:
    """
    Two 7-qubit blocks:
      control: 0..6
      target : 7..13
    ancillas: 14..19 (3 for control, 3 for target)

    Flow:
      - reset
      - prepare control in |1111111> (X-only "logical 1" demo)
      - inject X on control qubit 2 (malicious error)
      - optional X_ERROR(p_noise) on all data
      - transversal CNOT (control->target)
      - post-EC syndrome extraction (X-only) on both blocks
      - measure data qubits
    """
    c = stim.Circuit()
    control = list(range(0, 7))
    target  = list(range(7, 14))

    anc_c = [14, 15, 16]
    anc_t = [17, 18, 19]

    c.append("R", list(range(0, 20)))

    # Prepare control as |1111111> (a valid Hamming codeword)
    c.append("X", control)

    # Inject one malicious physical X
    c.append("X", [2])

    # Optional i.i.d. X-only noise on all 14 data qubits
    if p_noise > 0:
        c.append("X_ERROR", control + target, p_noise)

    # Transversal CNOT
    ops = []
    for qc, qt in zip(control, target):
        ops += [qc, qt]
    c.append("CX", ops)

    # Post-EC (X-only) on both blocks
    append_x_only_syndrome_extraction(c, control, anc_c)
    append_x_only_syndrome_extraction(c, target,  anc_t)

    # Measure data qubits
    c.append("M", control + target)
    return c

def decode_single_error_from_syndrome(bits7: np.ndarray, syndrome3: np.ndarray) -> np.ndarray:
    """Single-error Hamming decoder (t=1)."""
    s = tuple(int(x) for x in syndrome3.tolist())
    pos = S2I.get(s, None)
    out = bits7.copy()
    if pos is not None:
        out[pos] ^= 1
    return out

def parity(bits: np.ndarray) -> int:
    return int(bits.sum() % 2)

# ----------------------------
# Build circuit + display diagram
# ----------------------------
p_noise = 0.01
shots = 20_000

c_demo = build_two_block_demo_circuit(p_noise=p_noise)

display(c_demo.diagram('timeline-svg'))

# ----------------------------
# Run + decode
# ----------------------------
sampler = c_demo.compile_sampler(seed=12345)
ms = sampler.sample(shots).astype(np.uint8)

# Measurement layout:
# post-EC ancillas: 6 bits (3 for control + 3 for target)
# data: 14 bits
post_syn  = ms[:, 0:6]
data_meas = ms[:, 6:20]

control_data = data_meas[:, 0:7]
target_data  = data_meas[:, 7:14]

control_syn_post = post_syn[:, 0:3]
target_syn_post  = post_syn[:, 3:6]

decoded_control = np.empty_like(control_data)
decoded_target  = np.empty_like(target_data)

for i in range(shots):
    decoded_control[i] = decode_single_error_from_syndrome(control_data[i], control_syn_post[i])
    decoded_target[i]  = decode_single_error_from_syndrome(target_data[i],  target_syn_post[i])

# Expected parities in this demo:
# - control was prepared as all-ones -> parity 1
# - target should receive that via CNOT -> parity 1
expected_control_parity = 1
expected_target_parity = 1

control_par = np.array([parity(decoded_control[i]) for i in range(shots)], dtype=np.uint8)
target_par  = np.array([parity(decoded_target[i])  for i in range(shots)], dtype=np.uint8)

logical_fail = (control_par != expected_control_parity) | (target_par != expected_target_parity)

print("\n=== Results (post-decoding) ===")
print(f"p_noise={p_noise}, shots={shots}")
print(f"Logical failure rate (parity demo) = {logical_fail.mean():.6e}")


In [ ]:
import numpy as np
import stim
from IPython.display import display

# ----------------------------
# Hamming(7,4,3) Z-parity checks (detect X errors)
# ----------------------------
CHECKS = [
    [0, 2, 4, 6],
    [1, 2, 5, 6],
    [3, 4, 5, 6],
]

H = np.array([
    [1,0,1,0,1,0,1],
    [0,1,1,0,0,1,1],
    [0,0,0,1,1,1,1],
], dtype=np.uint8)

def make_syndrome_to_bitpos_map(H: np.ndarray):
    """Map a 3-bit syndrome to a single-bit error position (0..6)."""
    m, n = H.shape
    assert (m, n) == (3, 7)
    s2i = {tuple([0]*m): None}
    for j in range(n):
        s2i[tuple(H[:, j].tolist())] = j
    return s2i

S2I = make_syndrome_to_bitpos_map(H)

def append_x_only_syndrome_extraction(c: stim.Circuit, data: list[int], anc: list[int]):
    """
    Measure 3 Z-parity checks to detect X errors.
    Ancillas are reset, entangled via CX(data->anc), then measured in Z.
    """
    assert len(data) == 7 and len(anc) == 3
    c.append("R", anc)
    for r, qs in enumerate(CHECKS):
        a = anc[r]
        ops = []
        for q in qs:
            ops += [data[q], a]
        c.append("CX", ops)
    c.append("M", anc)

def build_two_block_demo_circuit(
    p_noise: float = 0.0,
    inject_error: bool = True,
    injected_qubit: int = 2,
) -> stim.Circuit:
    """
    Two 7-qubit blocks:
      control: 0..6
      target : 7..13
    ancillas: 14..19 (3 for control, 3 for target)

    Flow:
      - reset
      - prepare control in |1111111> (a valid Hamming codeword)
      - optionally inject a deterministic X on control[injected_qubit]
      - optionally apply i.i.d. X_ERROR(p_noise) on all 14 data qubits
      - transversal CNOT (control->target)
      - post-EC syndrome extraction (X-only) on both blocks
      - measure data qubits
    """
    c = stim.Circuit()
    control = list(range(0, 7))
    target  = list(range(7, 14))

    anc_c = [14, 15, 16]
    anc_t = [17, 18, 19]

    c.append("R", list(range(0, 20)))

    # Prepare control as |1111111>
    c.append("X", control)

    # Optional single injected X error on the control block
    if inject_error:
        if injected_qubit < 0 or injected_qubit > 6:
            raise ValueError("injected_qubit must be in [0..6] (control block index).")
        c.append("X", [injected_qubit])

    # Optional i.i.d. X-only noise on all 14 data qubits
    if p_noise > 0:
        c.append("X_ERROR", control + target, p_noise)

    # Transversal CNOT
    ops = []
    for qc, qt in zip(control, target):
        ops += [qc, qt]
    c.append("CX", ops)

    # Post-EC (X-only) on both blocks
    append_x_only_syndrome_extraction(c, control, anc_c)
    append_x_only_syndrome_extraction(c, target,  anc_t)

    # Measure data qubits
    c.append("M", control + target)
    return c

def decode_single_error_from_syndrome(bits7: np.ndarray, syndrome3: np.ndarray) -> np.ndarray:
    """Single-error Hamming decoder (t=1)."""
    s = tuple(int(x) for x in syndrome3.tolist())
    pos = S2I.get(s, None)
    out = bits7.copy()
    if pos is not None:
        out[pos] ^= 1
    return out

def parity(bits: np.ndarray) -> int:
    return int(bits.sum() % 2)

def run_and_score(c: stim.Circuit, shots: int, seed: int = 12345) -> float:
    """
    Runs the circuit, decodes both blocks, and returns the parity-demo failure rate.
    """
    sampler = c.compile_sampler(seed=seed)
    ms = sampler.sample(shots).astype(np.uint8)

    # Layout:
    # post-EC ancillas: 6 bits (3 control + 3 target)
    # data: 14 bits
    post_syn  = ms[:, 0:6]
    data_meas = ms[:, 6:20]

    control_data = data_meas[:, 0:7]
    target_data  = data_meas[:, 7:14]

    control_syn_post = post_syn[:, 0:3]
    target_syn_post  = post_syn[:, 3:6]

    decoded_control = np.empty_like(control_data)
    decoded_target  = np.empty_like(target_data)

    for i in range(shots):
        decoded_control[i] = decode_single_error_from_syndrome(control_data[i], control_syn_post[i])
        decoded_target[i]  = decode_single_error_from_syndrome(target_data[i],  target_syn_post[i])

    # Parity demo expectations:
    # control prepared as all-ones -> parity 1
    # target should receive that via CNOT -> parity 1
    expected_control_parity = 1
    expected_target_parity  = 1

    control_par = np.array([parity(decoded_control[i]) for i in range(shots)], dtype=np.uint8)
    target_par  = np.array([parity(decoded_target[i])  for i in range(shots)], dtype=np.uint8)

    logical_fail = (control_par != expected_control_parity) | (target_par != expected_target_parity)
    return float(logical_fail.mean())

# ----------------------------
# Two scenarios requested
# ----------------------------
shots = 20_000
p_noise = 0.01
injected_qubit = 2

# (A) Only injected error, NO i.i.d noise
c_injected_only = build_two_block_demo_circuit(
    p_noise=0.0,
    inject_error=True,
    injected_qubit=injected_qubit,
)

# (B) Only i.i.d noise, NO injected error
c_noise_only = build_two_block_demo_circuit(
    p_noise=p_noise,
    inject_error=False,
)

print("=== Scenario A: injected-only (single X), no i.i.d. noise ===")
display(c_injected_only.diagram("timeline-svg"))
rate_A = run_and_score(c_injected_only, shots=shots, seed=12345)
print(f"Logical failure rate (parity demo) = {rate_A:.6e}\n")

print("=== Scenario B: noise-only (X_ERROR), no injected error ===")
display(c_noise_only.diagram("timeline-svg"))
rate_B = run_and_score(c_noise_only, shots=shots, seed=23456)
print(f"Logical failure rate (parity demo) = {rate_B:.6e}")

In [ ]:
import numpy as np
import stim
from IPython.display import display

def append_Z_stabilizers(c: stim.Circuit, data: list[int], ancZ: list[int]):
    """
    Measure Z-type stabilizers (detect X errors) using ancillas in |0>.
    Circuit: R anc; CX(data->anc) for each 1; M anc
    """
    c.append("R", ancZ)
    for r, qs in enumerate(CHECKS):
        a = ancZ[r]
        ops = []
        for q in qs:
            ops += [data[q], a]
        c.append("CX", ops)
    c.append("M", ancZ)

def append_X_stabilizers(c: stim.Circuit, data: list[int], ancX: list[int]):
    """
    Measure X-type stabilizers (detect Z errors) using ancillas in |+> and MX readout.
    Circuit: R anc; H anc; CX(anc->data) for each 1; MX anc
    """
    c.append("R", ancX)
    c.append("H", ancX)  # prepare |+>
    for r, qs in enumerate(CHECKS):
        a = ancX[r]
        ops = []
        for q in qs:
            ops += [a, data[q]]  # CX(anc -> data)
        c.append("CX", ops)
    c.append("MX", ancX)

def build_H_demo_circuit(p_noise: float) -> stim.Circuit:
    """
    Prepare |0_L> by projecting with X stabilizers,
    apply X_ERROR(p) -> H^{⊗7} -> X_ERROR(p),
    measure Z and X stabilizers,
    then measure data in X basis (MX).
    """
    data = list(range(0, 7))
    ancZ = [7, 8, 9]
    ancX = [10, 11, 12]

    c = stim.Circuit()
    c.append("R", data + ancZ + ancX)

    # --- Prepare |0_L> via projection on X stabilizers ---
    append_X_stabilizers(c, data, ancX)  # 3 bits (prep)

    # --- Noise before the logical gate ---
    if p_noise > 0:
        c.append("X_ERROR", data, p_noise)

    # --- Logical H (transversal) ---
    c.append("H", data)

    # --- Noise after the logical gate ---
    if p_noise > 0:
        c.append("X_ERROR", data, p_noise)

    # --- EC syndrome extraction (both types) ---
    append_Z_stabilizers(c, data, ancZ)  # 3 bits (X-syndrome)
    append_X_stabilizers(c, data, ancX)  # 3 bits (Z-syndrome)

    # --- Measure logical-like observable in X basis ---
    c.append("MX", data)  # 7 bits

    return c

def run_H_demo(p_noise: float, shots: int, seed: int = 12345) -> float:
    c = build_H_demo_circuit(p_noise)
    display(c.diagram("timeline-svg"))

    sampler = c.compile_sampler(seed=seed)
    ms = sampler.sample(shots).astype(np.uint8)

    # Layout per shot:
    # prepX (3) | postZ (3) | postX (3) | dataX (7)  => total 16
    prepX   = ms[:, 0:3]
    postZ   = ms[:, 3:6]
    postX   = ms[:, 6:9]
    dataX   = ms[:, 9:16]

    fail = 0
    for i in range(shots):
        # Decode syndromes (t=1)
        x_corr_pos = S2I.get(tuple(int(b) for b in postZ[i].tolist()), None)  # would apply X on that qubit
        z_corr_pos = S2I.get(tuple(int(b) for b in postX[i].tolist()), None)  # would apply Z on that qubit

        # Measure X^{⊗7} via parity of MX outcomes.
        meas_par = parity(dataX[i])

        # For X-basis measurement: Z corrections flip outcomes, X corrections don't.
        corrected_par = meas_par ^ (1 if z_corr_pos is not None else 0)

        # Expected: +1 eigenvalue => parity 0
        if corrected_par != 0:
            fail += 1

    rate = fail / shots
    print(f"\n[H demo] p_noise={p_noise}, shots={shots}")
    print(f"Logical failure rate (X-parity after decoding) = {rate:.6e}")
    return rate

# --- Run ---
p_noise = 0.01
shots = 20_000
run_H_demo(p_noise=p_noise, shots=shots, seed=12345)

In [ ]:
import numpy as np
import stim
from IPython.display import display

def append_Z_stabilizers(c: stim.Circuit, data: list[int], ancZ: list[int]):
    c.append("R", ancZ)
    for r, qs in enumerate(CHECKS):
        a = ancZ[r]
        ops = []
        for q in qs:
            ops += [data[q], a]
        c.append("CX", ops)
    c.append("M", ancZ)

def append_X_stabilizers(c: stim.Circuit, data: list[int], ancX: list[int]):
    c.append("R", ancX)
    c.append("H", ancX)
    for r, qs in enumerate(CHECKS):
        a = ancX[r]
        ops = []
        for q in qs:
            ops += [a, data[q]]
        c.append("CX", ops)
    c.append("MX", ancX)

def build_S_demo_circuit(p_noise: float) -> stim.Circuit:
    """
    Prepare |0_L> by projection with X stabilizers.
    Prepare |+_L> by applying H^{⊗7}.
    Apply X_ERROR(p) -> S^{⊗7} -> X_ERROR(p).
    Measure Z and X stabilizers.
    Measure data in Y basis (MY), expecting +1 for Y^{⊗7} (parity 0).
    """
    data = list(range(0, 7))
    ancZ = [7, 8, 9]
    ancX = [10, 11, 12]

    c = stim.Circuit()
    c.append("R", data + ancZ + ancX)

    # --- Prepare |0_L> via projection on X stabilizers ---
    append_X_stabilizers(c, data, ancX)  # 3 bits (prep)

    # --- Prepare |+_L> (logical H transversal) ---
    c.append("H", data)

    # --- Noise before S ---
    if p_noise > 0:
        c.append("X_ERROR", data, p_noise)

    # --- Logical S (transversal) ---
    c.append("S", data)

    # --- Noise after S ---
    if p_noise > 0:
        c.append("X_ERROR", data, p_noise)

    # --- EC syndrome extraction (both types) ---
    append_Z_stabilizers(c, data, ancZ)  # 3 bits
    append_X_stabilizers(c, data, ancX)  # 3 bits

    # --- Measure logical-like observable in Y basis ---
    c.append("MY", data)  # 7 bits

    return c

def run_S_demo(p_noise: float, shots: int, seed: int = 23456) -> float:
    c = build_S_demo_circuit(p_noise)
    display(c.diagram("timeline-svg"))

    sampler = c.compile_sampler(seed=seed)
    ms = sampler.sample(shots).astype(np.uint8)

    # Layout per shot:
    # prepX (3) | postZ (3) | postX (3) | dataY (7)  => total 16
    prepX   = ms[:, 0:3]
    postZ   = ms[:, 3:6]
    postX   = ms[:, 6:9]
    dataY   = ms[:, 9:16]

    fail = 0
    for i in range(shots):
        x_corr_pos = S2I.get(tuple(int(b) for b in postZ[i].tolist()), None)  # X correction
        z_corr_pos = S2I.get(tuple(int(b) for b in postX[i].tolist()), None)  # Z correction

        meas_par = parity(dataY[i])

        # For Y-basis measurement: BOTH X and Z flip Y outcomes.
        # If both are present, flips cancel at the parity level (2 flips).
        flip = (1 if x_corr_pos is not None else 0) ^ (1 if z_corr_pos is not None else 0)
        corrected_par = meas_par ^ flip

        # Expected: +1 eigenvalue => parity 0
        if corrected_par != 0:
            fail += 1

    rate = fail / shots
    print(f"\n[S demo] p_noise={p_noise}, shots={shots}")
    print(f"Logical failure rate (Y-parity after decoding) = {rate:.6e}")
    return rate

# --- Run ---
p_noise = 0.01
shots = 20_000
run_S_demo(p_noise=p_noise, shots=shots, seed=23456)